In [0]:
from pyspark.sql.functions import col

df = spark.read.option("multiline", "true").json(
    "/Volumes/workspace/data/actually_fair_task/dataset_amazon-bestsellers-json.json"
)

df = df.withColumn(
    "price_value",
    col("price.value")
).withColumn(
    "currency",
    col("price.currency")
)

df.display()

In [0]:
df.printSchema()


In [0]:
from pyspark.sql.functions import col

df.filter(col("error").isNotNull()).display()

In [0]:
df = df.filter(col("error").isNull())
df.display()

In [0]:
df = df.drop(
    "thumbnailUrl",
    "input",
    "categoryUrl",
    "error",
    "price"
)

In [0]:
df.printSchema()

In [0]:
df = (
    df.withColumnRenamed("asin", "product_asin")
      .withColumnRenamed("name", "product_name")
      .withColumnRenamed("url", "product_url")
      .withColumnRenamed("position", "bestseller_rank")
      .withColumnRenamed("stars", "rating")
      .withColumnRenamed("reviewsCount", "review_count")
      .withColumnRenamed("numberOfOffers", "offers_count")
      .withColumnRenamed("categoryName", "category_name")
      .withColumnRenamed("categoryFullName", "category_full_name")
)

In [0]:
df.limit(5).display()

In [0]:
for column in df.columns:
    print(column, df.filter(col(column).isNull()).count())

In [0]:
df = df.dropna(subset=[
    "bestseller_rank",
    "price_value",
    "product_url"
])

In [0]:
df = df.fillna({
    "offers_count": 0,
    "review_count": 0,
    "rating": 0
})

In [0]:
for column in df.columns:
    print(column, df.where(col(column).isNull()).count())

In [0]:
df = df.dropDuplicates(["product_asin"])

In [0]:
df.display()

In [0]:
numeric_columns = [
    "offers_count",
    "bestseller_rank",
    "review_count",
    "rating",
    "price_value"
]

for column in numeric_columns:
    print(f"Checking column: {column}")
    
    df.select(column) \
      .distinct() \
      .orderBy(column) \
      .display()

In [0]:
from pyspark.sql.functions import col

df = (
    df.withColumn("offers_count", col("offers_count").cast("int"))
      .withColumn("bestseller_rank", col("bestseller_rank").cast("int"))
      .withColumn("review_count", col("review_count").cast("int"))
      .withColumn("rating", col("rating").cast("double"))
      .withColumn("price_value", col("price_value").cast("double"))
)

In [0]:
df.display()


In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import concat_ws

df = df.withColumn(
    "subcategories",
    concat_ws(", ", col("subcategories"))
)

In [0]:
df.display()

In [0]:
df = df.drop("subcategories")

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").parquet(
    "/Volumes/workspace/data/actually_fair_task/cleaned_amazon_bestsellers"
)

In [0]:
clean_df = spark.read.parquet(
    "/Volumes/workspace/data/actually_fair_task/cleaned_amazon_bestsellers"
)

display(clean_df)

In [0]:
df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/data/actually_fair_task/cleaned_amazon_bestsellers_csv")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn(
    "bestseller_score",
    (1 / col("bestseller_rank")) *
    log(col("review_count") + 1) *
    col("rating")
)

In [0]:
df = df.withColumn(
    "price_category",
    when(col("price_value") < 500, "Low")
    .when(col("price_value") < 4000, "Medium")
    .otherwise("High")
)

In [0]:
df = df.withColumn(
    "rating_bucket",
    when(col("rating") < 3, "Low")
    .when(col("rating") < 4, "Medium")
    .otherwise("High")
)

In [0]:
df = df.withColumn(
    "review_strength",
    when(col("review_count") < 100, "Weak")
    .when(col("review_count") < 1000, "Medium")
    .otherwise("Strong")
)

In [0]:


df = df.withColumn(
    "value_score",
    (col("rating") * log(col("review_count") + 1)) / (col("price_value") + 1)
)

In [0]:
df.display()

In [0]:
df.groupBy("category_name") \
  .avg("bestseller_score") \
  .orderBy("avg(bestseller_score)", ascending=False) \
  .display()

In [0]:
df.groupBy("category_name") \
  .avg("value_score") \
  .orderBy("avg(value_score)", ascending=False) \
  .display()

In [0]:
from pyspark.sql.functions import avg

df.groupBy("category_name") \
  .agg(
      avg("bestseller_score").alias("avg_bestseller_score"),
      avg("value_score").alias("avg_value_score")
  ) \
  .orderBy("avg_bestseller_score", ascending=False) \
  .display()

In [0]:
from pyspark.sql.functions import avg

df.groupBy("category_name") \
  .agg(
      avg("bestseller_score").alias("avg_bestseller_score"),
      avg("value_score").alias("avg_value_score")
  ) \
  .orderBy("avg_value_score", ascending=False) \
  .display()

In [0]:
df.display()

In [0]:
df=df.withColumn("combined_score",0.5*col("bestseller_score")+0.5*col("value_score"))


In [0]:
df.groupBy("category_name") \
  .agg(
      avg("combined_score").alias("avg_combined_score")
  ) \
  .orderBy(col("avg_combined_score").desc()) \
  .display()

In [0]:
df.printSchema()

In [0]:
df.groupBy("category_name") \
  .agg(avg("bestseller_score").alias("avg_bestseller_score")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.groupBy("category_name") \
  .agg(avg("value_score").alias("value_score")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.groupBy("category_name") \
  .agg(avg("combined_score").alias("avg_combined_score")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.select("rating", "bestseller_score").display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.select("rating").display()

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import avg, count

df.groupBy("category_name") \
  .agg(
      avg("bestseller_score").alias("avg_bestseller_score"),
      avg("value_score").alias("avg_value_score"),
      count("*").alias("product_count")
  ) \
  .orderBy("avg_bestseller_score", ascending=False) \
  .display()